# Stage 0 — Building the Dataset

**Event Recommendation System — Graduation Project (rebuild)**

We could not find a public event dataset with what this project needs: user interests, event descriptions, interaction history *and* 1–5 ratings. The best real candidate (Kaggle's 2013 *Event Recommendation Engine Challenge*) has no event text and no ratings, and its rules forbid sharing the data on GitHub.

So we **generate a synthetic dataset** — but not a random one. Random data has no patterns, so every model would score like a coin flip. Instead we simulate *people with real tastes* and let their behaviour produce the data.

> **Disclosure (also stated in the README):** the data is synthetic. Results show that the pipeline learns patterns correctly from data with known structure; they are not claims about real Egyptian event-goers.

## How the simulation works (the idea in plain words)

1. **Invent a world**: 8 categories → 24 sub-topics, each with its own keywords; 7 Egyptian cities.
2. **Give every user a hidden taste** that is *never saved* to the CSV files:
   - which categories they truly like (most people care about 2–3),
   - a hidden **style** (beginner↔advanced, calm↔energetic, budget↔premium),
   - their city, how willing they are to travel, how price-sensitive they are,
   - how active they are (most users are light users, a few are heavy users).
3. **Give every event hidden properties**: style and quality (good events become popular).
4. **Simulate behaviour**: each user browses events that fit them better *more often* (with randomness), then views → interested → going → attended, and rates attended events 1–5.
5. **Add a little realistic mess** (duplicates, missing values, typos) for the cleaning stage.

Why this gives *good but not perfect* results:

| Real-world effect we simulate | Which model can catch it |
|---|---|
| Category taste (stated interests are only ~85% honest) | Content-Based |
| Hidden style shared by similar users | Collaborative (SVD) — only partly visible in text |
| Popular high-quality events | Popularity baseline, SVD biases |
| Location and price | Hybrid / feature engineering |
| Random noise in every decision | Nobody — this keeps scores realistic |

Run the cells in order. The whole notebook takes a few seconds.

## 0. Setup
In Colab this downloads the project from GitHub and moves into it; locally it moves to the project root. The CSV files are written to `data/raw/`.

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/Abdelrahman2115/recommender-system.git"

if "google.colab" in sys.modules:                       # running inside Google Colab
    if not os.path.exists("/content/recommender-system"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/recommender-system"], check=True)
    os.chdir("/content/recommender-system")
elif os.path.basename(os.getcwd()) == "notebooks":      # running locally from the notebooks folder
    os.chdir("..")

sys.path.insert(0, "src")                               # make `import recsys` work
print("Project root:", os.getcwd())

Project root: /home/claude/recommender-system


## 1. Imports and global settings
- **`SEED`** makes the randomness *reproducible*: same seed → identical dataset every time. This is essential so anyone who reruns the notebook gets exactly the same data and results.
- `np.random.default_rng` is NumPy's modern random generator (better than the old `np.random.seed`).

In [2]:
# ---------------------------------------------------------------
# Stage 0 - Synthetic dataset generator for the Event Recommendation System
# ---------------------------------------------------------------
import numpy as np            # fast math on arrays (random numbers, vectors)
import pandas as pd           # tables (DataFrames) that we save as CSV
import os                     # create folders

SEED = 42                                  # fixed seed -> same dataset every run (reproducible)
rng = np.random.default_rng(SEED)          # modern NumPy random generator

N_USERS = 2000                             # number of users
N_EVENTS = 800                             # number of events
START = pd.Timestamp("2024-01-01")         # first day of the platform's history
END = pd.Timestamp("2025-12-31")           # last day of the platform's history

OUT_DIR = "data/raw"                       # where the CSV files will be written
os.makedirs(OUT_DIR, exist_ok=True)        # make the folder if it does not exist

## 2. Define the world: categories, keywords, cities, hidden style
- Each sub-topic has its own keywords. Events get their description from these words, which is what makes **TF-IDF** meaningful later: a "Python workshop" and a "data science meetup" share words, a "yoga" class does not.
- `CITY_BOOST` makes natural combinations more likely (diving events in Hurghada, heritage in Luxor).
- `STYLE_WORDS`: the hidden style only *sometimes* shows up in the text ("beginner-friendly", "premium"). That is realistic — descriptions never tell you everything — and it is the reason **collaborative filtering adds value** beyond content.

In [3]:
# ---------------------------------------------------------------
# 1) The "world": categories, sub-topics, and the words used to describe them
# ---------------------------------------------------------------
# Each category has sub-topics; each sub-topic has its own keywords.
# Keywords make event text meaningful, so TF-IDF has something real to learn.
CATEGORIES = {
    "Technology": {
        "AI & Data":        ["machine learning", "data science", "python", "neural networks", "analytics"],
        "Web & Mobile":     ["javascript", "react", "flutter", "web development", "apps"],
        "Cybersecurity":    ["security", "ethical hacking", "networks", "cloud", "privacy"],
    },
    "Arts & Culture": {
        "Painting":         ["painting", "canvas", "watercolor", "gallery", "sketching"],
        "Theatre & Film":   ["theatre", "cinema", "screening", "acting", "film"],
        "Heritage":         ["museum", "history", "pharaonic", "architecture", "heritage"],
    },
    "Sports & Fitness": {
        "Running":          ["running", "marathon", "5k", "endurance", "track"],
        "Football":         ["football", "five-a-side", "match", "league", "tournament"],
        "Yoga & Wellness":  ["yoga", "meditation", "stretching", "mindfulness", "wellness"],
    },
    "Food & Drink": {
        "Cooking Classes":  ["cooking", "recipes", "chef", "kitchen", "baking"],
        "Food Festivals":   ["street food", "festival", "tasting", "food trucks", "desserts"],
        "Coffee":           ["coffee", "specialty coffee", "barista", "brewing", "roasting"],
    },
    "Music": {
        "Live Concerts":    ["concert", "live band", "stage", "rock", "pop"],
        "Oriental Music":   ["oud", "tarab", "oriental", "classical arabic", "orchestra"],
        "Electronic":       ["dj", "electronic", "techno", "house", "beats"],
    },
    "Business & Career": {
        "Startups":         ["startup", "entrepreneurship", "pitching", "investors", "funding"],
        "Career Growth":    ["career", "cv", "interview skills", "networking", "leadership"],
        "Marketing":        ["marketing", "branding", "social media", "content", "sales"],
    },
    "Education & Workshops": {
        "Languages":        ["english", "german", "french", "language exchange", "conversation"],
        "Science":          ["science", "astronomy", "physics", "lecture", "research"],
        "Photography":      ["photography", "camera", "editing", "portrait", "lightroom"],
    },
    "Outdoors & Travel": {
        "Hiking & Camping": ["hiking", "camping", "desert", "trail", "stars"],
        "Diving & Sea":     ["diving", "snorkeling", "red sea", "beach", "coral"],
        "City Tours":       ["walking tour", "old cairo", "sightseeing", "guided tour", "markets"],
    },
}
CAT_NAMES = list(CATEGORIES.keys())        # the 8 category names in a fixed order
N_CATS = len(CAT_NAMES)

# Egyptian cities with a share of the population (Cairo is the biggest market).
CITIES = ["Cairo", "Giza", "Alexandria", "Mansoura", "Tanta", "Hurghada", "Luxor"]
CITY_P = np.array([0.40, 0.20, 0.18, 0.08, 0.05, 0.05, 0.04])

# Some categories are naturally linked to some cities (diving -> Hurghada, heritage -> Luxor).
CITY_BOOST = {("Outdoors & Travel", "Hurghada"): 3.0, ("Outdoors & Travel", "Luxor"): 1.5,
              ("Arts & Culture", "Luxor"): 2.0, ("Technology", "Cairo"): 1.5,
              ("Business & Career", "Cairo"): 1.5}

VENUES = ["co-working space", "hall", "outdoor", "cafe", "university", "online"]

# Hidden "style" dimensions: things that drive taste but are NOT columns in the data.
#   0: beginner (-) vs advanced (+)
#   1: calm (-)     vs energetic / social (+)
#   2: budget (-)   vs premium (+)
STYLE_DIM = 3
STYLE_WORDS = {  # words that sometimes leak the style into the text (only partially observable)
    (0, -1): ["beginner-friendly", "introductory", "no experience needed"],
    (0, +1): ["advanced", "deep dive", "for professionals"],
    (1, -1): ["relaxed", "small group", "calm atmosphere"],
    (1, +1): ["high energy", "meet new people", "big crowd"],
    (2, -1): ["affordable", "free entry", "student discount"],
    (2, +1): ["premium", "exclusive", "vip"],
}
GENERIC = ["join us", "community", "weekend", "experience", "learn", "fun", "friends",
           "session", "evening", "event", "hosted by", "special", "egypt"]

## 3. Generate events
Key lines:
- `rng.dirichlet(...)` → random **shares that sum to 1**; used so some categories have more events than others.
- `style` and `quality` are the event's **hidden** properties. They go into `ev_hidden`, which is *not* saved.
- **Price** uses a log-normal distribution (most events are cheap, a few are expensive) and is 0 for 25% of events — like real event platforms. Premium-style events cost more.
- 15% of descriptions borrow a word from another category, so categories are not perfectly separable.

In [4]:
# ---------------------------------------------------------------
# 2) Generate EVENTS
# ---------------------------------------------------------------
def make_event_text(cat, sub, style, rng):
    """Build a title and description from the sub-topic keywords + style words + noise."""
    kws = CATEGORIES[cat][sub]
    main = rng.choice(kws)                                   # the main keyword of the event
    title_tpl = rng.choice(["{k} meetup", "{k} workshop", "{k} night", "intro to {k}",
                            "{k} session", "{k} gathering", "{k} masterclass"])
    title = title_tpl.format(k=main).title()

    words = list(rng.choice(kws, size=3, replace=False))     # 3 topic keywords
    # Style words appear only when the hidden style is strong (and only 60% of the time),
    # so the text gives a hint about the hidden taste but never the full truth.
    for d in range(STYLE_DIM):
        if abs(style[d]) > 0.6 and rng.random() < 0.6:
            words.append(rng.choice(STYLE_WORDS[(d, int(np.sign(style[d])))]))
    # 15% chance to borrow a keyword from another category (real events mix topics).
    if rng.random() < 0.15:
        other_cat = rng.choice([c for c in CAT_NAMES if c != cat])
        other_sub = rng.choice(list(CATEGORIES[other_cat].keys()))
        words.append(rng.choice(CATEGORIES[other_cat][other_sub]))
    words += list(rng.choice(GENERIC, size=3, replace=False))  # generic filler words
    rng.shuffle(words)
    desc = f"{title}. A {sub.lower()} event about " + ", ".join(words) + "."
    return title, desc, kws

cat_share = rng.dirichlet(np.full(N_CATS, 5.0))              # some categories have more events
ev_rows, ev_hidden = [], []
for eid in range(1, N_EVENTS + 1):
    cat = CAT_NAMES[rng.choice(N_CATS, p=cat_share)]
    sub = rng.choice(list(CATEGORIES[cat].keys()))
    # city: population share, boosted for natural category-city pairs
    w = np.array([CITY_P[i] * CITY_BOOST.get((cat, c), 1.0) for i, c in enumerate(CITIES)])
    city = rng.choice(CITIES, p=w / w.sum())
    style = rng.normal(0, 1, STYLE_DIM).clip(-2, 2)           # hidden event style
    quality = rng.normal(0, 1)                                # hidden quality -> popularity
    title, desc, kws = make_event_text(cat, sub, style, rng)
    tags = "|".join(rng.choice(kws, size=2, replace=False))
    venue = rng.choice(VENUES, p=[0.15, 0.25, 0.15, 0.2, 0.15, 0.10])
    # price in EGP: free sometimes; otherwise log-normal, higher for "premium" style
    price = 0.0 if rng.random() < 0.25 else round(float(np.exp(rng.normal(5.3 + 0.35 * style[2], 0.6))), -1)
    start = START + pd.Timedelta(days=int(rng.integers(30, (END - START).days)),
                                 hours=int(rng.choice([10, 12, 16, 18, 19, 20])))
    organizer = int(rng.integers(1, 120))
    ev_rows.append(dict(event_id=eid, title=title, description=desc, category=cat, tags=tags,
                        city=city, venue_type=venue, start_datetime=start, price=price,
                        organizer_id=organizer))
    ev_hidden.append(dict(event_id=eid, cat_idx=CAT_NAMES.index(cat), style=style, quality=quality))

events = pd.DataFrame(ev_rows)
E_cat = np.array([h["cat_idx"] for h in ev_hidden])          # hidden arrays used only by the generator
E_style = np.vstack([h["style"] for h in ev_hidden])
E_quality = np.array([h["quality"] for h in ev_hidden])
E_city = events["city"].to_numpy()
E_start = events["start_datetime"].to_numpy()

## 4. Generate users with hidden tastes
- `rng.dirichlet(np.full(8, 0.35))`: a small parameter (0.35) produces *peaky* preferences — most of a user's taste sits in 1–3 categories. That matches how people actually behave.
- `U_leniency` is the hidden rating habit. It comes from a *separate* random generator so adding it did not change any other part of the data.
- `U_activity` uses a **log-normal** distribution → the "long tail": the median user has ~19 interactions, a few have 200+.
- `stated_interests()` creates the profile text the user typed. It is **deliberately imperfect**: true favourites are skipped 15% of the time and 30% of users add an "aspirational" interest they never act on. This is why a pure content model cannot be perfect.

In [5]:
# ---------------------------------------------------------------
# 3) Generate USERS (with hidden true tastes)
# ---------------------------------------------------------------
# True category preference: Dirichlet with small alpha -> each user cares about a few categories.
U_pref = rng.dirichlet(np.full(N_CATS, 0.35), size=N_USERS)
U_style = rng.normal(0, 1, (N_USERS, STYLE_DIM))              # hidden style taste
U_city = rng.choice(CITIES, size=N_USERS, p=CITY_P)
U_travel = rng.beta(2, 6, N_USERS)                            # willingness to go to other cities
U_price_sens = rng.gamma(2.0, 0.5, N_USERS)                   # how much price hurts
# Activity follows a long tail: most users do little, a few do a lot.
U_activity = np.clip(np.round(rng.lognormal(2.9, 0.8, N_USERS)), 3, 250).astype(int)
# Rating habit: some people rate generously, others harshly (a classic real-world effect).
# Drawn from a separate generator so it does not change any other part of the data.
U_leniency = np.random.default_rng(SEED + 7).normal(0, 0.45, N_USERS)

def stated_interests(pref, rng):
    """What the user TYPES in their profile: related to true taste but imperfect."""
    order = np.argsort(-pref)
    chosen = [CAT_NAMES[i] for i in order[:3] if pref[i] > 0.12 and rng.random() < 0.85]
    if not chosen:                                            # everyone picks at least one
        chosen = [CAT_NAMES[order[0]]]
    if rng.random() < 0.30:                                   # 30%: add a random "aspirational" interest
        chosen.append(rng.choice(CAT_NAMES))
    return "|".join(dict.fromkeys(chosen))                    # remove duplicates, keep order

users = pd.DataFrame({
    "user_id": np.arange(1, N_USERS + 1),
    "age": np.clip(rng.normal(27, 7, N_USERS).round(), 16, 65).astype(int),
    "city": U_city,
    "interests": [stated_interests(p, rng) for p in U_pref],
    # signup spread over the period, with more signups later (the platform grows)
    "signup_date": START + pd.to_timedelta((rng.beta(1.3, 1.0, N_USERS) * 600).astype(int), unit="D"),
})

## 5. Simulate interactions (the core of the generator)
The function `affinity(u)` is the hidden "truth" — how much user *u* would like each event:

`affinity = category taste + style match + event quality + location effect − price effect`

Then for each user:
1. Only events that **start after the user signed up** are possible (no time travel).
2. **Gumbel noise trick**: adding Gumbel noise to scores and taking the top *n* is the same as *sampling* events with probability proportional to `exp(affinity)` (a softmax). Better-fitting events are picked more often, but not always.
3. **Engagement funnel**: compared with the other events this user browsed, a better fit is more likely to become *interested*, *going* or *attended*.
4. **Rating** (only for attended events) = the user's **rating habit** (some people are generous, some harsh — `U_leniency`) + the event's **quality** + how well it **fits** this user + **noise**, rounded and clipped to 1–5. The noise (`normal(0, 0.6)`) means even a perfect model cannot predict ratings exactly — this sets a realistic lower limit on RMSE.
5. Timestamps are consistent: views happen *before* the event starts, ratings *after* it.

In [6]:
# ---------------------------------------------------------------
# 4) Generate INTERACTIONS from the hidden tastes
# ---------------------------------------------------------------
price_norm = np.log1p(events["price"].to_numpy()) / np.log1p(events["price"].max())

def affinity(u):
    """Hidden 'true' score of every event for user u (the thing a recommender tries to learn)."""
    cat_part = 2.2 * np.log(U_pref[u, E_cat] + 0.02)          # category taste (strongest signal)
    style_part = 0.55 * (E_style @ U_style[u])                # hidden style match (CF must discover it)
    qual_part = 0.6 * E_quality                               # good events attract everyone
    same_city = (E_city == U_city[u])
    city_part = np.where(same_city, 0.0, np.log(U_travel[u] + 0.02))  # other cities are less likely
    price_part = -1.2 * U_price_sens[u] * price_norm          # expensive events hurt price-sensitive users
    online = (events["venue_type"].to_numpy() == "online")
    city_part = np.where(online, 0.0, city_part)              # online events ignore distance
    return cat_part + style_part + qual_part + city_part + price_part

rows = []
for u in range(N_USERS):
    signup = users.loc[u, "signup_date"]
    # a user can only see events that start after they joined
    available = np.where(E_start > np.datetime64(signup + pd.Timedelta(days=3)))[0]
    if len(available) < 5:
        continue
    aff = affinity(u)[available]
    # exposure: the user browses events with probability ~ softmax(affinity) + random noise
    noisy = aff + rng.gumbel(0, 1.0, len(available))          # Gumbel noise = sampling from softmax
    n = min(U_activity[u], len(available))
    picked_local = np.argsort(-noisy)[:n]                     # top-n noisy scores = sampled events
    pa = aff[picked_local]                                    # affinities of the events this user browsed
    for j in picked_local:
        e = available[j]
        # funnel: compared with the OTHER events this user browsed, a better fit -> deeper engagement
        z = (aff[j] - pa.mean()) / (pa.std() + 1e-9)
        p_going = 1 / (1 + np.exp(-1.6 * (z - 0.9)))          # only clearly good fits get "going"
        p_interested = 1 / (1 + np.exp(-1.6 * (z - 0.1)))
        p_attend = 0.80                                       # 80% of "going" actually attend
        if rng.random() < p_going:
            itype = "attended" if rng.random() < p_attend else "going"
        elif rng.random() < p_interested:
            itype = "interested"
        else:
            itype = "view"
        start = pd.Timestamp(E_start[e])
        if itype == "attended":
            ts = start + pd.Timedelta(hours=int(rng.integers(3, 72)))   # rated after attending
            # rating = user's rating habit + event quality + how well it fits this user + noise
            rating = int(np.clip(np.round(2.8 + U_leniency[u] + 0.45 * E_quality[e] + 0.6 * z
                                          + rng.normal(0, 0.6)), 1, 5))
        else:
            lead = int(rng.integers(1, 25))
            ts = max(start - pd.Timedelta(days=lead), signup + pd.Timedelta(hours=1))
            ts += pd.Timedelta(minutes=int(rng.integers(0, 1440)))
            rating = np.nan                                   # no rating if not attended
        rows.append((u + 1, e + 1, ts, itype, rating))

interactions = pd.DataFrame(rows, columns=["user_id", "event_id", "timestamp",
                                           "interaction_type", "rating"])
interactions = interactions[interactions["timestamp"] <= END].sort_values("timestamp").reset_index(drop=True)

## 6. Add realistic data-quality problems
Real datasets are messy, and Stage 4 (cleaning) should have real work to do. We add a **small, documented** amount of mess:

| Problem | Amount | Fixed in Stage 4 by |
|---|---|---|
| Duplicate interaction rows | 1% | `drop_duplicates` |
| Missing ages | 3% | imputation (median) |
| Impossible ages (0, 3, 150, 999…) | 8 rows | treat as missing |
| City spelling (`cairo`, ` GIZA `) | 2% of users | strip + title-case |
| Missing description / price | 12 / 10 events | fill from title / median |

Note we use a **separate random generator** (`mess_rng`) so the mess doesn't change the rest of the data.

In [7]:
# ---------------------------------------------------------------
# 5) Add realistic data-quality problems (fixed later in Stage 4)
# ---------------------------------------------------------------
# Real data is never clean. We add a SMALL, documented amount of mess:
mess_rng = np.random.default_rng(SEED + 1)

# a) 1% duplicated interaction rows (e.g., a double click logged twice)
dup = interactions.sample(frac=0.01, random_state=SEED)
interactions = pd.concat([interactions, dup]).sort_values("timestamp").reset_index(drop=True)

# b) 3% missing ages and a few impossible ages (typing mistakes)
users["age"] = users["age"].astype("float")
users.loc[users.sample(frac=0.03, random_state=SEED).index, "age"] = np.nan
users.loc[users.sample(n=8, random_state=SEED + 2).index, "age"] = [3, 150, 999, 0, 7, 120, 210, 1]

# c) inconsistent city spelling / spacing in a few rows
bad = users.sample(frac=0.02, random_state=SEED + 3).index
users.loc[bad, "city"] = users.loc[bad, "city"].map(lambda c: mess_rng.choice([c.lower(), c.upper(), f" {c} "]))

# d) a few events with missing description or missing price
events.loc[events.sample(n=12, random_state=SEED).index, "description"] = np.nan
events.loc[events.sample(n=10, random_state=SEED + 4).index, "price"] = np.nan

## 7. Save the CSV files
Only the *observable* columns are saved. The hidden tastes (`U_pref`, `U_style`, `E_style`, `E_quality`) stay in memory and are thrown away — if we saved them and a model used them, that would be **data leakage** (the model would be reading the answer key).

In [8]:
# ---------------------------------------------------------------
# 6) Save the three CSV files (hidden tastes are NOT saved -> no leakage)
# ---------------------------------------------------------------
users.to_csv(f"{OUT_DIR}/users.csv", index=False)
events.to_csv(f"{OUT_DIR}/events.csv", index=False)
interactions.to_csv(f"{OUT_DIR}/interactions.csv", index=False)
print("users:", users.shape, "| events:", events.shape, "| interactions:", interactions.shape)
print(interactions["interaction_type"].value_counts())

users: (2000, 5) | events: (800, 10) | interactions: (50518, 5)
interaction_type
view          22174
interested    14692
attended      10861
going          2791
Name: count, dtype: int64


In [9]:
# ---------------------------------------------------------------
# 7) Quick look at what we produced (full EDA comes in Stage 3)
# ---------------------------------------------------------------
display(users.head())
display(events[["event_id", "title", "category", "city", "price", "start_datetime"]].head())
display(interactions.head())

,user_id,age,city,interests,signup_date
0,1,30.0,Giza,Food & Drink|Arts & Culture,2025-03-31
1,2,22.0,Alexandria,Food & Drink|Sports & Fitness|Education & Workshops,2024-12-02
2,3,29.0,Cairo,Arts & Culture|Sports & Fitness,2025-07-14
3,4,16.0,Cairo,Music|Outdoors & Travel|Arts & Culture,2025-05-05
4,5,26.0,Giza,Education & Workshops|Music,2024-05-15


,event_id,title,category,city,price,start_datetime
0,1,Language Exchange Gathering,Education & Workshops,Alexandria,260.0,2025-07-21 19:00:00
1,2,Brewing Gathering,Food & Drink,Cairo,120.0,2024-12-01 10:00:00
2,3,Ethical Hacking Night,Technology,Cairo,80.0,2024-10-23 20:00:00
3,4,Apps Workshop,Technology,Tanta,0.0,2025-07-14 16:00:00
4,5,Live Band Masterclass,Music,Cairo,120.0,2024-09-19 16:00:00


,user_id,event_id,timestamp,interaction_type,rating
0,312,226,2024-01-19 23:58:00,view,NaN
1,511,242,2024-01-21 09:08:00,view,NaN
2,470,72,2024-01-26 08:22:00,view,NaN
3,1276,170,2024-01-26 11:03:00,view,NaN
4,470,614,2024-01-26 18:02:00,going,NaN


## Data card (copy this into the README)

| File | Rows (approx.) | Columns |
|---|---|---|
| `users.csv` | 2,000 | user_id, age, city, interests, signup_date |
| `events.csv` | 800 | event_id, title, description, category, tags, city, venue_type, start_datetime, price, organizer_id |
| `interactions.csv` | ~50,500 | user_id, event_id, timestamp, interaction_type, rating |

- **Period:** 2024-01-01 → 2025-12-31 · **Cities:** 7 Egyptian cities · **Categories:** 8
- **Sparsity:** ~97% of user–event pairs have no interaction
- **interaction_type:** view, interested, going, attended · **rating:** 1–5, only for `attended`
- **Synthetic**, generated with seed 42 by `00_generate_dataset.ipynb`; contains intentional data-quality issues listed above.

### Questions an interviewer may ask you
1. *Why synthetic data?* — No public event dataset had interests, descriptions and ratings together; we simulated behaviour from hidden tastes so the data has learnable structure, and we disclose it.
2. *Isn't it circular — you designed the patterns, then found them?* — We designed **how people behave**, not the models. The models only see the CSVs, never the hidden tastes, and noise limits how well any model can do. The point is to test that the pipeline works correctly.
3. *How did you prevent leakage?* — Hidden variables are never saved; later we split by time and build all features from training data only.